# InternMatch AI

## Problématique
Comment recommander à un étudiant les offres d'emploi ou de stage les plus adaptées à son profil à partir de ses compétences et du contenu des offres ?

## Objectifs
- Charger et explorer les données
- Fusionner les informations sur les offres, compétences et descriptions
- Nettoyer les valeurs manquantes
- Créer une feature textuelle `matching_text`
- Transformer le texte avec TF-IDF
- Mesurer la similarité avec un profil utilisateur
- Retourner les meilleures recommandations


## 1. Imports et chemin des données


In [ ]:
import pandas as pd
import re
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
# Fonctionne si le notebook est lancé depuis le dossier du projet
# ou depuis le sous-dossier notebooks.
cwd = Path.cwd()

if (cwd / "data").exists():
    data_path = cwd / "data"
elif (cwd.parent / "data").exists():
    data_path = cwd.parent / "data"
else:
    raise FileNotFoundError(
        "Dossier data introuvable. Ouvre le notebook depuis le projet InternMatch-AI."
    )

data_path


## 2. Chargement des trois datasets


In [ ]:
df = pd.read_csv(data_path / "job_postings.csv")
skills = pd.read_csv(data_path / "job_skills.csv")
summary = pd.read_csv(data_path / "job_summary.csv")


In [ ]:
df.head()


In [ ]:
df.shape


In [ ]:
df.columns


In [ ]:
skills.columns


In [ ]:
summary.columns


## 3. Fusion des datasets


In [ ]:
df_merged = df.merge(
    skills,
    on="job_link",
    how="left"
)

df_merged = df_merged.merge(
    summary,
    on="job_link",
    how="left"
)


In [ ]:
df_merged.shape


In [ ]:
df_merged.columns


In [ ]:
df_merged.isnull().sum()


## 4. Sélection des colonnes utiles et nettoyage


In [ ]:
df_clean = df_merged[[
    "job_title",
    "company",
    "job_location",
    "job_level",
    "job_type",
    "job_skills",
    "job_summary",
    "job_link"
]].copy()


In [ ]:
df_clean = df_clean.dropna(
    subset=["job_skills", "job_location"]
)


In [ ]:
df_clean.shape


In [ ]:
df_clean.isnull().sum()


## 5. Observation du texte avant transformation


In [ ]:
df_clean["job_skills"].iloc[0]


In [ ]:
df_clean["job_summary"].iloc[0]


## 6. Feature engineering : création de `matching_text`


In [ ]:
df_clean["matching_text"] = (
    df_clean["job_title"] + " " +
    df_clean["job_skills"] + " " +
    df_clean["job_summary"]
)


In [ ]:
df_clean["matching_text"].iloc[0]


## 7. Nettoyage du texte


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+#. ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [ ]:
df_clean["matching_text"] = (
    df_clean["matching_text"].apply(clean_text)
)


In [ ]:
df_clean["matching_text"].iloc[0]


## 8. Transformation TF-IDF


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

job_vectors = vectorizer.fit_transform(
    df_clean["matching_text"]
)


In [ ]:
job_vectors.shape


## 9. Similarité entre un profil et les offres


In [ ]:
profile = '''
Data science student looking for an internship.
Skills: Python, SQL, pandas, machine learning,
data analysis and data visualization.
'''


In [ ]:
profile_clean = clean_text(profile)
profile_vector = vectorizer.transform([profile_clean])

scores = cosine_similarity(
    profile_vector,
    job_vectors
).flatten()


## 10. Top recommandations


In [ ]:
results = df_clean.copy()
results["match_score"] = (scores * 100).round(1)

top_jobs = results.sort_values(
    by="match_score",
    ascending=False
).head(5)


In [ ]:
top_jobs[[
    "job_title",
    "company",
    "job_location",
    "job_skills",
    "match_score"
]]


## 11. Fonction finale de recommandation


In [ ]:
def recommend_jobs(profile, top_n=5):
    profile_clean = clean_text(profile)
    profile_vector = vectorizer.transform([profile_clean])

    scores = cosine_similarity(
        profile_vector,
        job_vectors
    ).flatten()

    results = df_clean.copy()
    results["match_score"] = (scores * 100).round(1)

    return results.sort_values(
        by="match_score",
        ascending=False
    )[[
        "job_title",
        "company",
        "job_location",
        "job_level",
        "job_type",
        "job_skills",
        "match_score",
        "job_link"
    ]].head(top_n)


## 12. Tests


In [ ]:
profile_data = '''
Data science student.
Skills: Python, SQL, pandas, machine learning,
statistics and data visualization.
Looking for a data analyst or data science internship.
'''

recommend_jobs(profile_data)


In [ ]:
profile_web = '''
Computer science student.
Skills: JavaScript, React, HTML, CSS,
Node.js and web development.
Looking for a frontend developer internship.
'''

recommend_jobs(profile_web)
